In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def prepare_data(
    destr_file,
    spot_curve_file,
    short_rate_column='DESTR',
    spot_columns=['1Y', '5Y', '10Y', '20Y', '30Y'],
    spot_maturities=[1, 5, 10, 20, 30],
    frequency='M'  # 'M' = end-of-month; 'W' = weekly; 'BMS' = beginning of month
):
    """
    Load and align short rate (e.g. DESTR) and spot curve data.

    Parameters:
        destr_file : str
            Path to the short-rate CSV or Excel file.
        spot_curve_file : str
            Path to the spot rate CSV or Excel file.
        short_rate_column : str
            Name of the short rate column (e.g. 'DESTR').
        spot_columns : list
            List of column names for the spot curve.
        spot_maturities : list
            Corresponding maturities (in years).
        frequency : str
            Pandas resample frequency (e.g. 'M', 'W', 'BMS').

    Returns:
        pd.DataFrame
            Merged dataframe with date, short rate, and zero-coupon prices.
    """
    # Load short rate (DESTR)
    destr = pd.read_csv(destr_file, parse_dates=True, index_col=0)
    destr = destr[[short_rate_column]].resample(frequency).last()

    # Load spot rates
    spot = pd.read_csv(spot_curve_file, parse_dates=True, index_col=0)
    spot = spot[spot_columns].resample(frequency).last()

    # Align dates
    merged = destr.join(spot, how='inner')

    # Convert spot rates to zero-coupon bond prices
    for col, tau in zip(spot_columns, spot_maturities):
        price_col = f'ZCB_{tau}Y'
        merged[price_col] = np.exp(-merged[col] * tau)

    # Final output: keep short rate and ZCB prices
    zcb_columns = [f'ZCB_{tau}Y' for tau in spot_maturities]
    final = merged[[short_rate_column] + zcb_columns].dropna()

    return final

In [ ]:
if __name__ == '__main__':
    destr_file = 'destr.csv'           # Your DESTR daily CSV with columns: ['Date', 'DESTR']
    spot_curve_file = 'spot_curve.csv' # ECB spot rates CSV: ['Date', '1Y', '5Y', '10Y', ...]

    df = prepare_data(destr_file, spot_curve_file)
    print(df.head())